# MedGemma Clinical NER — Colab Runner (T4 free tier)

Zero-shot evaluation of **google/medgemma-4b-it** on the combined
**NCBI Disease + BC5CDR** test set (harmonized to Disease/Chemical), scored with
seqeval — the *same test set and scorer* as the sibling `clinical-ner-eval` repo,
so the resulting `results/comparison.csv` concatenates directly with its results.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`.

MedGemma is a **gated** model: accept the license at
https://huggingface.co/google/medgemma-4b-it and log in with an HF token (cell 2).

## 1. Set the T4 GPU runtime and confirm it

`Runtime -> Change runtime type -> Hardware accelerator: T4 GPU`, then run:

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU. Set Runtime -> Change runtime type -> T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))

## 2. Install dependencies

In [ ]:
# datasets pinned <3.0 so bigbio trust_remote_code loading scripts still work.
!pip install -q -U transformers accelerate bitsandbytes "datasets>=2.20,<3.0" \
    bioc seqeval pandas tqdm huggingface_hub

## 3. Hugging Face login (gated model)

Paste a token from https://huggingface.co/settings/tokens when prompted. The
token is read at runtime via `getpass` and never stored in the notebook.

In [ ]:
from getpass import getpass
from huggingface_hub import login
login(getpass('HF token (input hidden): '))

## 4. Get the project code

Clone your repo (or upload the `medgemma-ner-eval` folder to the Colab file tree
and skip the clone). Then `cd` into it so `python -m src.evaluate` resolves.

In [ ]:
# !git clone <YOUR_REPO_URL> medgemma-ner-eval
%cd medgemma-ner-eval

## 5. Smoke test first (`--limit 10`)

Runs the full pipeline (load model -> prompt -> parse -> align -> seqeval) on 10
examples. Confirms the model loads under 4-bit and the JSON/alignment path works
before committing to the full ~600-sentence run. Overwrites `results/comparison.csv`
with a 10-example table — the full run in the next cell replaces it.

In [ ]:
!python -m src.evaluate --limit 10

## 6. Full evaluation

Runs on the entire combined NCBI + BC5CDR test set (~1,363 gold entities) and
writes `results/comparison.csv` + `results/full_report.json`.

In [ ]:
!python -m src.evaluate

## 7. Inspect results and combine with the sibling repo

In [ ]:
import pandas as pd
med = pd.read_csv('results/comparison.csv')
print(med.to_string(index=False))

### Combining with `clinical-ner-eval`

Both repos emit identical columns (`model, entity, precision, recall, f1, support`)
over the identical test set and scorer, so combining is a concatenation:

```python
import pandas as pd
med = pd.read_csv('results/comparison.csv')
bert = pd.read_csv('../clinical-ner-eval/results/comparison.csv')
combined = pd.concat([bert, med], ignore_index=True)
combined.to_csv('combined_comparison.csv', index=False)
print(combined[combined.entity == 'micro avg'])
```